# M07 — Build a Reusable Data-to-Model Pipeline

Build, interrogate, break, repair, cross-validate and persist one CPU-only raw-data-to-prediction pipeline. This notebook uses only the local synthetic fixture and contains no learner answers or prefilled execution output.

## Mission contract

The public boundary is raw rows in and predictions out. Split before fitting learned preprocessing, keep the entire pipeline inside cross-validation, exclude identifiers/targets/post-outcome fields, and reuse the fitted transformations at inference.

In [ ]:
from pathlib import Path
import sys
import tempfile

import numpy as np
import pandas as pd
from sklearn.exceptions import NotFittedError
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.utils.validation import check_is_fitted

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'missions' / 'M07').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from missions.M07.pipeline import (
    CATEGORICAL_FEATURES, MODEL_FEATURES, NUMERIC_FEATURES,
    build_pipeline, load_dataset, load_pipeline, save_pipeline,
    split_features_target, train_test_frames, transform_features,
)

DATASET_PATH = PROJECT_ROOT / 'datasets' / 'M07' / 'customer_renewals.csv'
RANDOM_STATE = 17

## Inspect the raw contract

**Predict before running:** Which columns will be excluded, and which numerical and categorical columns contain missing values? Record your prediction before the next cell.

In [ ]:
frame = load_dataset(DATASET_PATH)
X, y = split_features_target(frame)
summary = {
    'rows': len(frame),
    'model_features': list(X.columns),
    'excluded_columns': sorted(set(frame.columns) - set(X.columns)),
    'missing_by_feature': X.isna().sum().to_dict(),
    'target_counts': y.value_counts().sort_index().to_dict(),
}
assert tuple(X.columns) == MODEL_FEATURES
assert set(summary['excluded_columns']) == {'customer_id', 'renewed'}
summary

## Leakage boundary: split raw rows first

Imputers, scalers, category vocabularies and model coefficients all learn state. The split below happens before any of them are fitted. The feature allow-list also rejects identifiers, the target, and undeclared post-outcome columns.

In [ ]:
augmented = frame.assign(post_outcome_contact=frame['renewed'])
safe_X, safe_y = split_features_target(augmented)
assert 'post_outcome_contact' not in safe_X
assert 'customer_id' not in safe_X
assert 'renewed' not in safe_X
X_train, X_test, y_train, y_test = train_test_frames(frame, random_state=RANDOM_STATE)
split_summary = {'train_rows': len(X_train), 'test_rows': len(X_test), 'train_classes': y_train.value_counts().to_dict()}
split_summary

## Experiment 1 — Manual vs pipeline

**Predict before running:** Which approach has more independently managed state and a larger train/inference mismatch surface? Scores are secondary; record a prediction about operational behavior before executing both paths.

In [ ]:
manual_medians = X_train[list(NUMERIC_FEATURES)].median()
manual_modes = X_train[list(CATEGORICAL_FEATURES)].mode().iloc[0]

def manual_prepare(raw, expected_columns=None):
    prepared = raw.copy()
    prepared.loc[:, list(NUMERIC_FEATURES)] = prepared[list(NUMERIC_FEATURES)].fillna(manual_medians)
    prepared.loc[:, list(CATEGORICAL_FEATURES)] = prepared[list(CATEGORICAL_FEATURES)].fillna(manual_modes)
    matrix = pd.get_dummies(prepared, columns=list(CATEGORICAL_FEATURES), dtype=float)
    if expected_columns is not None:
        matrix = matrix.reindex(columns=expected_columns, fill_value=0.0)
    return matrix

manual_train = manual_prepare(X_train)
manual_test = manual_prepare(X_test, expected_columns=manual_train.columns)
manual_model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE).fit(manual_train, y_train)
manual_accuracy = accuracy_score(y_test, manual_model.predict(manual_test))
{'manual_accuracy': manual_accuracy, 'manual_matrix_width': manual_train.shape[1], 'separate_state_objects': ['medians', 'modes', 'dummy_columns', 'model']}

In [ ]:
pipeline = build_pipeline(random_state=RANDOM_STATE)
pipeline.fit(X_train, y_train)
pipeline_accuracy = accuracy_score(y_test, pipeline.predict(X_test))
feature_names = pipeline.named_steps['preprocess'].get_feature_names_out()
comparison = {
    'manual_accuracy': manual_accuracy,
    'pipeline_accuracy': pipeline_accuracy,
    'pipeline_matrix_width': len(feature_names),
    'public_inference_input': list(X_test.columns),
}
assert set(pipeline.named_steps) == {'preprocess', 'model'}
comparison

## fit vs transform

`fit` learns imputation statistics, scale parameters, category vocabularies and model coefficients. `transform` applies only the fitted preprocessing; `predict` applies that preprocessing and then the fitted model. Repeated inference must not mutate learned state.

In [ ]:
unfitted = build_pipeline()
try:
    transform_features(unfitted, X_test)
except NotFittedError as exc:
    prefit_error = type(exc).__name__
else:
    raise AssertionError('transform unexpectedly succeeded before fit')

numeric_branch = pipeline.named_steps['preprocess'].named_transformers_['numeric']
mean_before = numeric_branch.named_steps['scale'].mean_.copy()
first_matrix = transform_features(pipeline, X_test)
second_matrix = transform_features(pipeline, X_test)
np.testing.assert_allclose(first_matrix, second_matrix)
np.testing.assert_allclose(mean_before, numeric_branch.named_steps['scale'].mean_)
{'before_fit': prefit_error, 'after_fit_shape': first_matrix.shape, 'state_unchanged': True}

## Experiment 2 — Change feature treatment

**Predict before running:** With this small fixture, will robust scaling materially change held-out accuracy relative to standard scaling? Hold the split, imputation strategy and estimator fixed.

In [ ]:
robust_pipeline = build_pipeline(scaler='robust', random_state=RANDOM_STATE)
robust_pipeline.fit(X_train, y_train)
robust_accuracy = accuracy_score(y_test, robust_pipeline.predict(X_test))
treatment_result = {'standard_accuracy': pipeline_accuracy, 'robust_accuracy': robust_accuracy, 'delta': robust_accuracy - pipeline_accuracy}
treatment_result

## Experiment 3 — Cross-validation through the pipeline

**Predict before running:** Will `cross_validate` fit the original object or clones? Passing raw training rows and the complete pipeline keeps every fold's imputer, scaler and encoder inside that fold's training boundary.

In [ ]:
cv_pipeline = build_pipeline(random_state=RANDOM_STATE)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = cross_validate(cv_pipeline, X_train, y_train, cv=cv, scoring='accuracy', return_estimator=True)
try:
    check_is_fitted(cv_pipeline.named_steps['preprocess'])
except NotFittedError:
    original_remained_unfitted = True
else:
    original_remained_unfitted = False
assert original_remained_unfitted
cv_summary = {'fold_scores': cv_results['test_score'].round(3).tolist(), 'mean': float(cv_results['test_score'].mean()), 'original_remained_unfitted': original_remained_unfitted}
cv_summary

## Experiment 4 — Serialize and reload

**Predict before running:** Will the reloaded object produce identical transformations, probabilities and class predictions? Joblib artifacts must come from trusted sources and should travel with dependency-version metadata.

In [ ]:
expected_matrix = transform_features(pipeline, X_test)
expected_predictions = pipeline.predict(X_test)
expected_probabilities = pipeline.predict_proba(X_test)
with tempfile.TemporaryDirectory(prefix='m07-notebook-') as directory:
    artifact_path = save_pipeline(pipeline, Path(directory) / 'renewal_pipeline.joblib')
    reloaded_pipeline = load_pipeline(artifact_path)
np.testing.assert_allclose(expected_matrix, transform_features(reloaded_pipeline, X_test))
np.testing.assert_array_equal(expected_predictions, reloaded_pipeline.predict(X_test))
np.testing.assert_allclose(expected_probabilities, reloaded_pipeline.predict_proba(X_test))
reload_result = {'transforms_identical': True, 'predictions_identical': True, 'probabilities_identical': True}
reload_result

## Experiment 5 — Identical inference transformations

**Predict before running:** An inference row contains categories absent from training. Will the fitted pipeline keep the same transformed width, and what information is lost when unknown categories are ignored?

In [ ]:
unseen_row = X_test.iloc[[0]].copy()
unseen_row.loc[:, 'plan'] = 'enterprise'
unseen_row.loc[:, 'region'] = 'central'
unseen_row.loc[:, 'signup_channel'] = 'partner'
training_width = transform_features(pipeline, X_train.iloc[[0]]).shape[1]
unseen_matrix = transform_features(pipeline, unseen_row)
unseen_prediction = pipeline.predict(unseen_row)
assert unseen_matrix.shape == (1, training_width)
parity_result = {'training_width': training_width, 'inference_width': unseen_matrix.shape[1], 'prediction': int(unseen_prediction[0])}
parity_result

## Controlled failure — independent train/inference encoding

The next experiment intentionally recreates the fragile manual boundary: `get_dummies` is called separately on training and unseen-category inference rows. The expected exception is captured so Restart + Run All succeeds.

### Predict before running

Record the expected training width, inference width, exception family and root cause. Do not repair the row by deleting its new categories.

In [ ]:
broken_train = manual_prepare(X_train)
broken_inference = manual_prepare(unseen_row)
broken_model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE).fit(broken_train, y_train)
try:
    broken_model.predict(broken_inference)
except ValueError as exc:
    controlled_error = {'type': type(exc).__name__, 'message': str(exc), 'train_width': broken_train.shape[1], 'inference_width': broken_inference.shape[1]}
else:
    raise AssertionError('controlled mismatch unexpectedly produced a prediction')
assert tuple(broken_train.columns) != tuple(broken_inference.columns)
controlled_error

### Diagnose before repairing

Compare column names and shapes. The estimator is behaving correctly: independently learned dummy schemas disagree. The smallest boundary repair is to pass the raw inference row through the fitted pipeline, not to refit on inference data.

In [ ]:
only_in_training = sorted(set(broken_train.columns) - set(broken_inference.columns))
only_in_inference = sorted(set(broken_inference.columns) - set(broken_train.columns))
repaired_matrix = transform_features(pipeline, unseen_row)
repaired_prediction = pipeline.predict(unseen_row)
assert repaired_matrix.shape[1] == training_width
repair_evidence = {'only_in_training_sample': only_in_training[:4], 'only_in_inference': only_in_inference, 'repaired_width': repaired_matrix.shape[1], 'repaired_prediction': int(repaired_prediction[0])}
repair_evidence

## Experiment summary

Explain the observations rather than treating accuracy as the only result: manual versus pipeline state, the isolated scaler change, fold-local preprocessing, reload equivalence, unknown-category semantics, and the controlled mismatch repair.

In [ ]:
verification = {
    'manual_vs_pipeline': comparison,
    'feature_treatment': treatment_result,
    'cross_validation': cv_summary,
    'reload': reload_result,
    'inference_parity': parity_result,
    'controlled_failure_repaired': repaired_matrix.shape[1] == training_width,
}
assert all(reload_result.values())
assert verification['controlled_failure_repaired']
verification

## Formal engineering review

Use `missions/M07/review_brief.md`. Defend the raw-row interface, feature availability assumptions, fold boundary, unknown-category policy, trusted serialization policy, tests and residual uncertainty. A high fixture score is not approval evidence.

## ADR

Complete `missions/M07/adr_prompt.md` using `templates/ADR.md`. Compare manual duplication, separately persisted preprocessing and one pipeline artifact. Record consequences, operational controls, rollback triggers and evidence that could change the decision.

## No-AI transfer gate

Complete `missions/M07/no_ai_gate.md` on a fresh mixed-type table without AI-generated code. Learner evidence is intentionally not prefilled in this notebook.